# Train ACT on `yaditya/qual-cheese-salami-final-model` (Colab)

ACT (Action Chunking Transformer) has **no language input**, so the 8 task labels
(`pick-n-put-{cheese|salami} put-in-q{1..4}`) are fed as a one-hot vector in
`observation.environment_state`. The order is saved in `task_vocab.json` next to the weights;
`follow.py` / `kitchen_qual.py` read it and build the one-hot from the task string.

1. **Runtime → Change runtime type → GPU** (T4 works; L4/A100 is faster).
2. Run the cells top to bottom. If Colab asks to restart after the install, restart and continue from cell 2.
3. Download `act-qual-policy.zip`, unzip into the repo root (next to `toppings-v2`), then:

```bash
.venv/bin/python kitchen_qual.py --checkpoint ../../act-qual-policy --bus-serial 5B61032837 \
  --motor-ids 1,2,3,4,5,6,7,8 --camera-order 1,0 --max-delta-ticks 1500
```

The dataset is public; no Hugging Face login is needed unless you push the model.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "lerobot==0.3.3"

## Settings

In [ ]:
import os

REPO_ID = "yaditya/qual-cheese-salami-final-model"
OUT = "act-qual-policy"              # folder that kitchen_qual.py --checkpoint points at
STEPS = int(os.environ.get("ACT_STEPS", 30_000))
BATCH = int(os.environ.get("ACT_BATCH", 8))
LR = 1e-5                            # ACT paper / LeRobot default
CHUNK = 20                           # actions predicted per inference (2 s at 10 fps)
N_ACTION_STEPS = 10                  # actions executed before re-planning (1 s); smaller = more reactive
SAVE_EVERY = 5_000
WORKERS = int(os.environ.get("ACT_WORKERS", 2))
AUGMENT = True                       # brightness/contrast/saturation/hue/sharpness jitter: robust to lighting
# Canonical task order. Never reorder after training: the one-hot index is the task identity.
TASKS = [f"pick-n-put-{t} put-in-q{q}" for t in ("cheese", "salami") for q in (1, 2, 3, 4)]

## Dataset

In [ ]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.datasets.transforms import ImageTransforms, ImageTransformsConfig

meta = LeRobotDatasetMetadata(REPO_ID)
FPS = meta.fps
print(f"{meta.total_episodes} episodes, {meta.total_frames} frames at {FPS} fps")
print("features:", {k: v["shape"] for k, v in meta.features.items() if k.startswith(("observation", "action"))})
dataset_tasks = set(meta.tasks.values())
unknown = dataset_tasks - set(TASKS)
assert not unknown, f"Dataset has tasks outside TASKS: {unknown}"
print("tasks without data:", sorted(set(TASKS) - dataset_tasks) or "none")

transforms = ImageTransforms(ImageTransformsConfig(enable=True, max_num_transforms=3, random_order=True)) if AUGMENT else None
dataset = LeRobotDataset(
    REPO_ID,
    delta_timestamps={"action": [i / FPS for i in range(CHUNK)]},
    image_transforms=transforms,
    video_backend="pyav",
)
CAMERAS = sorted(k for k in meta.features if k.startswith("observation.images."))
assert CAMERAS == ["observation.images.camera_0", "observation.images.camera_1"], CAMERAS
sample = dataset[0]
print({k: tuple(v.shape) for k, v in sample.items() if hasattr(v, "shape")}, sample["task"])

counts = {}
for i in range(meta.total_episodes):
    task = meta.episodes[i]["tasks"][0]
    counts[task] = counts.get(task, 0) + 1
for task in TASKS:
    print(f"{counts.get(task, 0):3d} episodes  {task}")

## Policy
The task one-hot is an `ENV` feature, which ACT does not normalize, so it stays exactly 0/1.

In [ ]:
from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
features = dataset_to_policy_features(meta.features)
inputs = {k: f for k, f in features.items() if f.type is not FeatureType.ACTION}
inputs["observation.environment_state"] = PolicyFeature(FeatureType.ENV, (len(TASKS),))
outputs = {k: f for k, f in features.items() if f.type is FeatureType.ACTION}
config = ACTConfig(
    input_features=inputs, output_features=outputs, chunk_size=CHUNK, n_action_steps=N_ACTION_STEPS,
    optimizer_lr=LR, optimizer_lr_backbone=LR, device=str(device),
)
policy = ACTPolicy(config, dataset_stats=meta.stats).to(device).train()
print(f"{sum(p.numel() for p in policy.parameters()) / 1e6:.1f}M parameters")


def one_hot(tasks):
    hot = torch.zeros(len(tasks), len(TASKS), device=device)
    hot[torch.arange(len(tasks)), torch.tensor([TASKS.index(t) for t in tasks])] = 1
    return hot


def to_device(batch):
    out = {k: v.to(device, non_blocking=True) for k, v in batch.items() if isinstance(v, torch.Tensor)}
    out["observation.environment_state"] = one_hot(batch["task"])
    return out

## Train

In [ ]:
import json, time
from pathlib import Path


def save(path):
    path = Path(path)
    policy.save_pretrained(path)
    (path / "task_vocab.json").write_text(json.dumps(
        {"feature": "observation.environment_state", "tasks": TASKS}, indent=2))


loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH, shuffle=True, num_workers=WORKERS,
                                     pin_memory=device.type == "cuda", drop_last=True,
                                     persistent_workers=WORKERS > 0)
optimizer = torch.optim.AdamW(policy.get_optim_params(), lr=LR, weight_decay=config.optimizer_weight_decay)
step, start, history = 0, time.time(), []
while step < STEPS:
    for batch in loader:
        loss, info = policy.forward(to_device(batch))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 10.0)
        optimizer.step()
        step += 1
        history.append(loss.item())
        if step % 100 == 0 or step == STEPS:
            recent = sum(history[-100:]) / len(history[-100:])
            print(f"step {step}/{STEPS}  loss {recent:.4f}  l1 {info.get('l1_loss', 0):.4f}  "
                  f"{(time.time() - start) / step:.2f}s/step", flush=True)
        if step % SAVE_EVERY == 0 and step < STEPS:
            save(f"{OUT}-step{step}")
        if step >= STEPS:
            break
save(OUT)
print("saved", OUT, sorted(p.name for p in Path(OUT).iterdir()))

## Open-loop check
Runs the policy on recorded frames of one episode per task and compares with the recorded actions. Actions are calibrated joint positions in [0, 1]. Curves should follow each other; large offsets on one joint mean more data or more steps.

In [ ]:
import matplotlib.pyplot as plt

policy.eval()
eval_set = LeRobotDataset(REPO_ID, video_backend="pyav")
episode = {meta.episodes[i]["tasks"][0]: i for i in range(meta.total_episodes)}
for task, ep in list(episode.items())[:2]:
    lo, hi = eval_set.episode_data_index["from"][ep].item(), eval_set.episode_data_index["to"][ep].item()
    policy.reset()
    pred, true = [], []
    with torch.no_grad():
        for i in range(lo, hi):
            frame = eval_set[i]
            batch = {k: v.unsqueeze(0) for k, v in frame.items() if isinstance(v, torch.Tensor) and k != "action"}
            batch["task"] = [frame["task"]]
            pred.append(policy.select_action(to_device(batch))[0].cpu())
            true.append(frame["action"])
    pred, true = torch.stack(pred), torch.stack(true)
    error = (pred - true).abs().mean(0) * 100  # actions are calibrated positions in [0, 1]
    print(f"{task}: mean |error| per joint, % of joint range: {[round(e, 1) for e in error.tolist()]}")
    fig, axes = plt.subplots(2, 4, figsize=(16, 5), sharex=True)
    for j, ax in enumerate(axes.flat):
        ax.plot(true[:, j], label="recorded"); ax.plot(pred[:, j], label="policy"); ax.set_title(f"joint {j + 1}")
    axes[0, 0].legend(); fig.suptitle(task); plt.show()
policy.train()

## Download
Unzip into the repo root so the folder is `robot-pizza-cup/act-qual-policy/` (contains `config.json`, `model.safetensors`, `task_vocab.json`).

In [ ]:
import shutil
shutil.make_archive(OUT, "zip", OUT)
try:
    from google.colab import files
    files.download(f"{OUT}.zip")
except ImportError:
    print("Not on Colab; zip is at", f"{OUT}.zip")

### Optional: keep a copy on Google Drive, or push to the Hub
Colab deletes files when the runtime stops.

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copytree(OUT, f"/content/drive/MyDrive/{OUT}", dirs_exist_ok=True)

# from huggingface_hub import login, upload_folder; login()
# upload_folder(repo_id="yaditya/act-qual-policy", folder_path=OUT, repo_type="model")